# 07 · 最终提交包与无网自测(★★★★★,技术不难但最易丢分)

确认:权重位置、推理入口、配置、源码、**无网 import、无绝对路径(提交用相对)、输出目录、文件名、压缩包结构、解压即跑**。

> 注意:本教程内部用绝对 BASE 方便你在 Jupyter 跑;**真正提交的包里要改成相对路径**(见下检查)。

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


## 1) 提交包应有的结构(参照官方)

In [ ]:
SUBMIT = f"{WORK}/submit_pangu_weather"
need = {
  "inference.py": "推理入口(读 conf/config.yaml,输出 result/output/*.npy)",
  "conf/config.yaml": "配置(test_ratio 改成官方要求年份)",
  "data/checkpoints/student.pth": "权重(或按 download_model_url 下载)",
  "maxvit3d_student.py": "学生模型定义(若用学生)",
  "result/output/": "输出目录(必须存在)",
}
for k,v in need.items(): print(f"  {k:34s} {v}")

## 2) 无网自测检查清单(逐条 assert)

In [ ]:
import ast, io, zipfile
checks = []
# a) 推理脚本里没有硬编码绝对路径(/public /work2 等)
def no_abs_path(pyfile):
    if not os.path.exists(pyfile): return None
    src = open(pyfile, encoding="utf-8").read()
    bad = [s for s in ["/public/", "/work2/", "/home/"] if s in src]
    return "无绝对路径" if not bad else f"⚠含绝对路径 {bad}"
print("推理脚本路径检查:", no_abs_path(f"{SUBMIT}/inference.py") or "(未找到inference.py,先放进去)")
# b) 计时区只包 model(x):检查 start_time/end_time 之间只有一行 model 调用(人工核对)
print("提示:手动确认计时区 start_time..end_time 之间只有 model(x) 一句")
# c) 输出目录存在
print("输出目录:", "存在" if os.path.isdir(f"{SUBMIT}/result/output") else "缺失(mkdir -p result/output)")

## 3) 打包成 zip + 校验结构(顶层目录、无 __pycache__、含权重与入口)

In [ ]:
import zipfile
# 演示:把 SUBMIT 打包(实战 SUBMIT 里要放齐 inference.py/conf/权重/源码)
if os.path.isdir(SUBMIT):
    zpath = f"{WORK}/提交包.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root,_,fs in os.walk(SUBMIT):
            for f in fs:
                if "__pycache__" in root or f.endswith(".pyc"): continue   # 别打进缓存
                fp = os.path.join(root,f)
                z.write(fp, os.path.relpath(fp, os.path.dirname(SUBMIT)))   # 顶层目录=包名
    with zipfile.ZipFile(zpath) as z: names = z.namelist()
    print("zip 顶层:", sorted(set(n.split("/")[0] for n in names)))
    print("含入口:", any(n.endswith("inference.py") for n in names), "| 含权重:", any(".pth" in n for n in names))
else:
    print("先建 SUBMIT 目录并放齐文件:", SUBMIT)

### ✅ 最易丢分点(逐条过):
1. 提交包里**用相对路径**(`./conf/config.yaml`),不要 `/public/...`。
2. 计时区 `start..end` 之间**只有 `model(x)`**。
3. 输出到 `result/output/`,文件名 = 目标时刻(如 `2050010100.npy`)。
4. **无网 import**:onescience/torch 都是容器自带;别 import 装不上的包。
5. 压缩包**顶层是一个目录**,不含 `__pycache__`/`.pyc`。
6. 解压到生成结果**能一条龙跑通**(自己解压到干净目录试一遍)。